# Netherlands and Germany: Aldi shelf prices in EUR

One run of
[Aldi Netherlands & Germany Price Scraper: Shelf Prices (EUR)](https://apify.com/yappman/nl-de-supermarket-price-scraper)
that reads two category pages in one go: fresh dairy at Aldi Netherlands (`aldi.nl`) and milk at Aldi Nord
Germany (`aldi-nord.de`). For each shop the notebook sorts the products by the shop's own unit price, within each measure, and
counts the products that show none.

**Cost at Apify Free-tier prices:** at most **$0.05**.

| Run | Rows (`maxItems`) | Rows | Start | Most it can cost |
|---|---|---|---|---|
| Category, two shops | 30 | $0.045 | $0.005 | $0.05 |

Each row is $0.0015 on the Free tier. `maxItems` is shared evenly between the two shops, so each returns at most 15 rows.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
import time
from datetime import timedelta
from decimal import Decimal
from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)
pd.set_option("display.html.use_mathjax", False)  # show third-party text as text

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    The run times out after 10 minutes.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
        run_timeout=timedelta(minutes=10),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    # Charges settle a few seconds after the run ends, so read the record again before printing them.
    time.sleep(5)
    settled = client.run(run.id).get()
    print("Charged events:", settled.charged_event_counts)
    return run

## 1. Read one category page at each shop

Category mode takes the shop from each page address, so no `retailers` list is needed. The run's `RUN_SUMMARY`
record says, per shop, whether it finished and how many rows it returned.

In [ ]:
ACTOR = "yappman/nl-de-supermarket-price-scraper"
PAGES = [
    "https://www.aldi.nl/producten/zuivel-eieren-boter/verse-zuivel.html",
    "https://www.aldi-nord.de/sortiment/milchprodukte/milch-milchgetraenke.html",
]

# Free tier: 30 rows x $0.0015 + 1 start x $0.005 = $0.05
run = run_actor(
    ACTOR,
    {"mode": "category", "categoryUrls": PAGES, "maxItems": 30},
    max_charge_usd=0.055,
)
df = pd.DataFrame(client.dataset(run.default_dataset_id).iterate_items())

summary = client.key_value_store(run.default_key_value_store_id).get_record("RUN_SUMMARY")
if summary:
    for shop, part in summary["value"].get("perRetailer", {}).items():
        print(f"{shop}: status {part.get('status')}, rows {part.get('pushed')}, unpriced {part.get('unpriced')}")
print(len(df), "rows")

## 2. Sort each shop by its own unit price

`unitPrice` is the shop's own figure (`value` per `measure`), and it is empty where the shop shows none: the Actor
never calculates one. Rows are sorted within each measure, because a price per kg and a price per litre
are not comparable. `reduced` is true while the shop shows a was price above the current price.

In [ ]:
df["unitValue"] = df["unitPrice"].apply(lambda u: (u or {}).get("value"))
df["unitMeasure"] = df["unitPrice"].apply(lambda u: (u or {}).get("measure"))
df["reduced"] = df["wasPrice"].notna() & (df["wasPrice"] > df["price"])
df["category"] = df["categoryPath"].apply(lambda p: p[-1] if p else None)

COLUMNS = ["name", "brand", "packSize", "price", "wasPrice", "unitValue", "unitMeasure", "reduced", "category"]
for shop, rows in df.groupby("retailer"):
    print(f"{shop}: {len(rows)} products, {rows['unitValue'].isna().sum()} with no unit price shown")
    rows.to_csv(OUTPUT / f"eu_{shop}_prices.csv", index=False)
    # Sort within each measure (kg, l, ...): a price per kg and a price per litre are not comparable.
    display(rows.sort_values(["unitMeasure", "unitValue"], na_position="last")[COLUMNS])

## Notes

* Prices are in euros, as shown to a visitor with no store chosen, at the time in `scrapedAt`.
* The two shops sell different ranges and name products in Dutch and German. This example compares products within
  each shop, not across them.
* Aldi shows no barcode, so `ean` is always empty. `matchKey` is a name and pack-size key.
* At aldi-nord.de the bottle deposit (Pfand) is not a field; `price` is the shelf price as the site gives it.
* Rows belong to the Actor's data terms, not to this repository's MIT licence.
  The CSVs are written to `output/`, which git ignores.